ICD-10 coding pipeline — Step 2: Multi-path ICD-10 candidate retrieval
*Co-authored with CoCo*

# Step 2: Multi-Path ICD-10 Candidate Retrieval

## How it works

For each extracted finding, we retrieve ICD-10 code candidates via **three independent retrieval paths**, then merge and rank them:

1. **Semantic Search (Path A):** Cortex Search with a structured query (finding + acuity + laterality + body_site + causal_link). Finds codes by meaning.
2. **Category-Constrained Search (Path B):** Narrow to a likely ICD-10 chapter based on heuristics, then rank by Jaro-Winkler string similarity. Catches codes with different terminology.
3. **HCC Expansion (Path C):** If Path A's top result has an HCC category, retrieve all other codes in that same HCC. Catches sibling codes the LLM might prefer for specificity.

## Scoring
- Each path contributes a score: `weight / rank` (semantic=1.0, category=0.7, HCC=0.5)
- Codes found by multiple paths get their scores summed → higher confidence
- Final output is top-N candidates per finding, ranked by combined score

## Key design decisions
- **Structured query, not raw quote:** Using extracted metadata (acuity, laterality, causal_link) in the search query improves precision
- **Category=ruled_out excluded:** Outpatient rule-outs should not be coded
- **Dual-code pre-fetch:** Known dagger/asterisk pairs are flagged for Step 3

In [ ]:
%%sql -r ctx
-- Context
USE ROLE ACCOUNTADMIN;
USE DATABASE ICD10_CODING_APP;
USE SCHEMA MATCHING;
USE WAREHOUSE COMPUTE_WH;
ALTER SESSION SET QUERY_TAG = 'icd10_v2:search';

---
## Configuration

In [ ]:
%%sql -r config
-- ============================================================
-- SEARCH CONFIG
-- ============================================================
SET SEMANTIC_K = 10;          -- Candidates from semantic search (Path A)
SET CATEGORY_K = 5;           -- Candidates from category search (Path B)
SET HCC_EXPANSION_K = 5;      -- Candidates from HCC sibling expansion (Path C)
SET FINAL_TOP_N = 15;         -- Final merged candidates per finding

-- Path weights for combined scoring
SET WEIGHT_SEMANTIC = 1.0;
SET WEIGHT_CATEGORY = 0.7;
SET WEIGHT_HCC = 0.5;

---
## UDF 1: Build Structured Search Query

Scalar UDF that takes extracted finding metadata and returns an optimized search query string.

In [ ]:
%%sql -r search_svc
-- UDF: Build structured search query from finding metadata
CREATE OR REPLACE FUNCTION MATCHING.BUILD_SEARCH_QUERY(
    FINDING VARCHAR,
    CATEGORY VARCHAR,
    ACUITY VARCHAR,
    LATERALITY VARCHAR,
    BODY_SITE VARCHAR,
    SEVERITY VARCHAR,
    CAUSAL_LINK VARCHAR
)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
    CONCAT(
        FINDING,
        CASE WHEN ACUITY IN ('acute', 'chronic', 'acute_on_chronic') THEN ' ' || ACUITY ELSE '' END,
        CASE WHEN LATERALITY NOT IN ('not_applicable', 'unspecified') THEN ' ' || LATERALITY ELSE '' END,
        CASE WHEN BODY_SITE IS NOT NULL THEN ' ' || BODY_SITE ELSE '' END,
        CASE WHEN SEVERITY NOT IN ('unspecified') THEN ' ' || SEVERITY ELSE '' END,
        CASE WHEN CAUSAL_LINK IS NOT NULL THEN ' due to ' || CAUSAL_LINK ELSE '' END
    )
$$;

---
## UDF 2: Classify Chapter Prefix

Scalar UDF that maps a finding to its likely ICD-10 chapter for category-constrained search (Path B).

In [ ]:
%%sql -r queries
-- UDF: Classify finding into ICD-10 chapter prefix
CREATE OR REPLACE FUNCTION MATCHING.GET_CHAPTER_PREFIX(
    FINDING VARCHAR,
    CATEGORY VARCHAR,
    CAUSAL_LINK VARCHAR
)
RETURNS VARCHAR
LANGUAGE SQL
AS
$$
    CASE
        WHEN CATEGORY = 'history_of' THEN 'Z'
        WHEN CATEGORY = 'symptom' THEN 'R'
        WHEN LOWER(CAUSAL_LINK) LIKE '%diabet%' THEN 'E1'
        WHEN LOWER(FINDING) LIKE '%diabetes%' THEN 'E1'
        WHEN LOWER(FINDING) LIKE '%hypertension%' OR LOWER(FINDING) LIKE '%heart%' THEN 'I'
        WHEN LOWER(FINDING) LIKE '%cancer%' OR LOWER(FINDING) LIKE '%carcinoma%' THEN 'C'
        WHEN LOWER(FINDING) LIKE '%pneumonia%' OR LOWER(FINDING) LIKE '%asthma%' OR LOWER(FINDING) LIKE '%copd%' THEN 'J'
        WHEN LOWER(FINDING) LIKE '%depression%' OR LOWER(FINDING) LIKE '%anxiety%' THEN 'F'
        WHEN LOWER(FINDING) LIKE '%kidney%' OR LOWER(FINDING) LIKE '%renal%' THEN 'N1'
        WHEN LOWER(FINDING) LIKE '%arthritis%' OR LOWER(FINDING) LIKE '%osteo%' THEN 'M'
        ELSE NULL
    END
$$;

---
## UDF 3: Score Candidate by Path

Scalar UDF that computes the weighted score for a candidate based on its retrieval path and rank.

In [ ]:
%%sql -r path_b
-- UDF: Compute path score for a candidate
CREATE OR REPLACE FUNCTION MATCHING.COMPUTE_PATH_SCORE(
    RETRIEVAL_PATH VARCHAR,
    CANDIDATE_RANK INTEGER,
    W_SEMANTIC FLOAT,
    W_CATEGORY FLOAT,
    W_HCC FLOAT
)
RETURNS FLOAT
LANGUAGE SQL
AS
$$
    CASE RETRIEVAL_PATH
        WHEN 'SEMANTIC' THEN W_SEMANTIC / CANDIDATE_RANK
        WHEN 'CATEGORY' THEN W_CATEGORY / CANDIDATE_RANK
        WHEN 'HCC_EXPANSION' THEN W_HCC / CANDIDATE_RANK
        ELSE 0
    END
$$;

---
## Run Pipeline

Executes the search pipeline using the UDFs above. Each step creates a table.

In [ ]:
%%sql -r step1_svc
-- Step 1: Create search service (run once or when reference data changes)
CREATE OR REPLACE CORTEX SEARCH SERVICE ICD10_REF.ICD10_SEARCH_SVC
    ON SEARCH_TEXT
    WAREHOUSE = COMPUTE_WH
    TARGET_LAG = '1 hour'
    AS (
        SELECT
            ICD10_CODE, SHORT_DESCRIPTION, LONG_DESCRIPTION, CATEGORY_CODE,
            CHAPTER, HCC_CATEGORY, HCC_DESCRIPTION, RAF_COEFFICIENT,
            IS_DUAL_CODE_ETIOLOGY, IS_DUAL_CODE_MANIFESTATION, SEARCH_TEXT
        FROM ICD10_REF.ICD10_CODES_ENRICHED
    );

In [ ]:
%%sql -r step2_queries
-- Step 2: Build search queries using UDFs
CREATE OR REPLACE TABLE MATCHING.SEARCH_QUERIES AS
SELECT
    FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, ACUITY, LATERALITY,
    BODY_SITE, SEVERITY, CAUSAL_LINK, SUPPORTING_QUOTE,
    MATCHING.BUILD_SEARCH_QUERY(FINDING, CATEGORY, ACUITY, LATERALITY, BODY_SITE, SEVERITY, CAUSAL_LINK) AS SEARCH_QUERY,
    MATCHING.GET_CHAPTER_PREFIX(FINDING, CATEGORY, CAUSAL_LINK) AS CHAPTER_PREFIX
FROM PROCESSING.ENCOUNTER_FINDINGS
WHERE CATEGORY NOT IN ('ruled_out');

In [ ]:
%%sql -r step3a
-- Step 3a: Path A - Semantic search
CREATE OR REPLACE TABLE MATCHING.CANDIDATES_SEMANTIC AS
SELECT
    q.FILE_NAME, q.FINDING_SEQ, q.FINDING, q.CATEGORY, q.CAUSAL_LINK,
    r.ICD10_CODE,
    r.SHORT_DESCRIPTION AS DESCRIPTION,
    r.LONG_DESCRIPTION,
    r.CHAPTER,
    TRY_CAST(r.HCC_CATEGORY AS NUMBER) AS HCC_CATEGORY,
    TRY_CAST(r.RAF_COEFFICIENT AS FLOAT) AS RAF_COEFFICIENT,
    TRY_CAST(r.IS_DUAL_CODE_ETIOLOGY AS BOOLEAN) AS IS_DUAL_CODE_ETIOLOGY,
    TRY_CAST(r.IS_DUAL_CODE_MANIFESTATION AS BOOLEAN) AS IS_DUAL_CODE_MANIFESTATION,
    r.METADATA$RANK AS CANDIDATE_RANK,
    'SEMANTIC' AS RETRIEVAL_PATH
FROM MATCHING.SEARCH_QUERIES q,
LATERAL CORTEX_SEARCH_BATCH(
    service_name => 'ICD10_CODING_APP.ICD10_REF.ICD10_SEARCH_SVC',
    query => q.SEARCH_QUERY,
    limit => $SEMANTIC_K
) AS r;

In [ ]:
%%sql -r step3b
-- Step 3b: Path B - Category-constrained search (Jaro-Winkler within chapter)
CREATE OR REPLACE TABLE MATCHING.CANDIDATES_CATEGORY AS
SELECT
    q.FILE_NAME, q.FINDING_SEQ, q.FINDING, q.CATEGORY, q.CAUSAL_LINK,
    c.ICD10_CODE,
    c.SHORT_DESCRIPTION AS DESCRIPTION,
    c.LONG_DESCRIPTION,
    c.CHAPTER,
    hcc.HCC_CATEGORY,
    hcc.RAF_COEFFICIENT,
    ce.IS_DUAL_CODE_ETIOLOGY,
    ce.IS_DUAL_CODE_MANIFESTATION,
    ROW_NUMBER() OVER (
        PARTITION BY q.FILE_NAME, q.FINDING_SEQ
        ORDER BY JAROWINKLER_SIMILARITY(LOWER(q.FINDING), LOWER(COALESCE(c.LONG_DESCRIPTION, c.SHORT_DESCRIPTION))) DESC
    ) AS CANDIDATE_RANK,
    'CATEGORY' AS RETRIEVAL_PATH
FROM MATCHING.SEARCH_QUERIES q
JOIN ICD10_REF.ICD10_CODES c ON c.ICD10_CODE LIKE q.CHAPTER_PREFIX || '%'
LEFT JOIN ICD10_REF.HCC_MAPPINGS hcc ON c.ICD10_CODE = hcc.ICD10_CODE
LEFT JOIN ICD10_REF.ICD10_CODES_ENRICHED ce ON c.ICD10_CODE = ce.ICD10_CODE
WHERE q.CHAPTER_PREFIX IS NOT NULL
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY q.FILE_NAME, q.FINDING_SEQ
    ORDER BY JAROWINKLER_SIMILARITY(LOWER(q.FINDING), LOWER(COALESCE(c.LONG_DESCRIPTION, c.SHORT_DESCRIPTION))) DESC
) <= $CATEGORY_K;

In [ ]:
%%sql -r step3c
-- Step 3c: Path C - HCC expansion (sibling codes in same HCC category)
CREATE OR REPLACE TABLE MATCHING.CANDIDATES_HCC_EXPANSION AS
WITH top_semantic AS (
    SELECT FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, CAUSAL_LINK,
           ICD10_CODE AS TOP_CODE, HCC_CATEGORY AS TOP_HCC
    FROM MATCHING.CANDIDATES_SEMANTIC
    WHERE CANDIDATE_RANK = 1 AND HCC_CATEGORY IS NOT NULL
)
SELECT
    ts.FILE_NAME, ts.FINDING_SEQ, ts.FINDING, ts.CATEGORY, ts.CAUSAL_LINK,
    c.ICD10_CODE,
    c.SHORT_DESCRIPTION AS DESCRIPTION,
    c.LONG_DESCRIPTION,
    c.CHAPTER,
    h.HCC_CATEGORY,
    h.RAF_COEFFICIENT,
    ce.IS_DUAL_CODE_ETIOLOGY,
    ce.IS_DUAL_CODE_MANIFESTATION,
    ROW_NUMBER() OVER (
        PARTITION BY ts.FILE_NAME, ts.FINDING_SEQ
        ORDER BY JAROWINKLER_SIMILARITY(LOWER(ts.FINDING), LOWER(c.SHORT_DESCRIPTION)) DESC
    ) AS CANDIDATE_RANK,
    'HCC_EXPANSION' AS RETRIEVAL_PATH
FROM top_semantic ts
JOIN ICD10_REF.HCC_MAPPINGS h ON h.HCC_CATEGORY = ts.TOP_HCC
JOIN ICD10_REF.ICD10_CODES c ON c.ICD10_CODE = h.ICD10_CODE
LEFT JOIN ICD10_REF.ICD10_CODES_ENRICHED ce ON c.ICD10_CODE = ce.ICD10_CODE
WHERE c.ICD10_CODE != ts.TOP_CODE
QUALIFY ROW_NUMBER() OVER (
    PARTITION BY ts.FILE_NAME, ts.FINDING_SEQ
    ORDER BY JAROWINKLER_SIMILARITY(LOWER(ts.FINDING), LOWER(c.SHORT_DESCRIPTION)) DESC
) <= $HCC_EXPANSION_K;

In [ ]:
%%sql -r step4_merge
-- Step 4: Merge all paths using scoring UDF, deduplicate, rank
CREATE OR REPLACE TABLE MATCHING.CANDIDATES_MERGED AS
WITH all_candidates AS (
    SELECT *, MATCHING.COMPUTE_PATH_SCORE(RETRIEVAL_PATH, CANDIDATE_RANK, $WEIGHT_SEMANTIC, $WEIGHT_CATEGORY, $WEIGHT_HCC) AS PATH_SCORE
    FROM MATCHING.CANDIDATES_SEMANTIC
    UNION ALL
    SELECT *, MATCHING.COMPUTE_PATH_SCORE(RETRIEVAL_PATH, CANDIDATE_RANK, $WEIGHT_SEMANTIC, $WEIGHT_CATEGORY, $WEIGHT_HCC) AS PATH_SCORE
    FROM MATCHING.CANDIDATES_CATEGORY
    UNION ALL
    SELECT *, MATCHING.COMPUTE_PATH_SCORE(RETRIEVAL_PATH, CANDIDATE_RANK, $WEIGHT_SEMANTIC, $WEIGHT_CATEGORY, $WEIGHT_HCC) AS PATH_SCORE
    FROM MATCHING.CANDIDATES_HCC_EXPANSION
),
deduped AS (
    SELECT
        FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, CAUSAL_LINK, ICD10_CODE,
        MAX(DESCRIPTION) AS DESCRIPTION,
        MAX(LONG_DESCRIPTION) AS LONG_DESCRIPTION,
        MAX(CHAPTER) AS CHAPTER,
        MAX(HCC_CATEGORY) AS HCC_CATEGORY,
        MAX(RAF_COEFFICIENT) AS RAF_COEFFICIENT,
        BOOLOR_AGG(IS_DUAL_CODE_ETIOLOGY) AS IS_DUAL_CODE_ETIOLOGY,
        BOOLOR_AGG(IS_DUAL_CODE_MANIFESTATION) AS IS_DUAL_CODE_MANIFESTATION,
        SUM(PATH_SCORE) AS COMBINED_SCORE,
        LISTAGG(DISTINCT RETRIEVAL_PATH, ',') AS FOUND_BY_PATHS,
        COUNT(DISTINCT RETRIEVAL_PATH) AS PATH_COUNT
    FROM all_candidates
    GROUP BY FILE_NAME, FINDING_SEQ, FINDING, CATEGORY, CAUSAL_LINK, ICD10_CODE
)
SELECT *,
    ROW_NUMBER() OVER (PARTITION BY FILE_NAME, FINDING_SEQ ORDER BY COMBINED_SCORE DESC) AS FINAL_RANK
FROM deduped
QUALIFY FINAL_RANK <= $FINAL_TOP_N;

In [ ]:
%%sql -r step5_dual
-- Step 5: Flag dual-code pair candidates
CREATE OR REPLACE TABLE MATCHING.DUAL_CODE_CANDIDATES AS
SELECT DISTINCT
    cm.FILE_NAME, cm.FINDING_SEQ, cm.FINDING, cm.CAUSAL_LINK,
    dp.ETIOLOGY_CODE, dp.MANIFESTATION_CODE, dp.RELATIONSHIP, dp.DESCRIPTION AS PAIR_DESCRIPTION,
    e_desc.SHORT_DESCRIPTION AS ETIOLOGY_DESCRIPTION,
    m_desc.SHORT_DESCRIPTION AS MANIFESTATION_DESCRIPTION
FROM MATCHING.CANDIDATES_MERGED cm
JOIN ICD10_REF.DUAL_CODE_PAIRS dp
    ON cm.ICD10_CODE IN (dp.ETIOLOGY_CODE, dp.MANIFESTATION_CODE)
LEFT JOIN ICD10_REF.ICD10_CODES e_desc ON dp.ETIOLOGY_CODE = e_desc.ICD10_CODE
LEFT JOIN ICD10_REF.ICD10_CODES m_desc ON dp.MANIFESTATION_CODE = m_desc.ICD10_CODE
WHERE cm.CAUSAL_LINK IS NOT NULL;

---
## Output Preview

In [ ]:
%%sql -r stats
-- Retrieval statistics
SELECT
    COUNT(DISTINCT FILE_NAME || '-' || FINDING_SEQ) AS FINDINGS_WITH_CANDIDATES,
    COUNT(*) AS TOTAL_CANDIDATES,
    ROUND(COUNT(*)::FLOAT / NULLIF(COUNT(DISTINCT FILE_NAME || '-' || FINDING_SEQ), 0), 1) AS AVG_PER_FINDING,
    COUNT_IF(PATH_COUNT > 1) AS MULTI_PATH_HITS,
    COUNT_IF(HCC_CATEGORY IS NOT NULL) AS HCC_RELEVANT
FROM MATCHING.CANDIDATES_MERGED;

In [ ]:
%%sql -r paths
-- Path contribution
SELECT FOUND_BY_PATHS, COUNT(*) AS CNT, ROUND(AVG(FINAL_RANK), 1) AS AVG_RANK
FROM MATCHING.CANDIDATES_MERGED
GROUP BY FOUND_BY_PATHS
ORDER BY CNT DESC;

---
## Done

**Output:** `MATCHING.CANDIDATES_MERGED` — top-N ranked ICD-10 candidates per finding

**Next:** Run `03_matching.ipynb` to select final codes via LLM.